# 30-day readmission prediction

UCI diabetes hospital dataset. Target is 1 if readmitted within 30 days, 0 otherwise.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve,
)
import joblib

DATA_PATH = os.path.join("data", "diabetic_data.csv")
OUTPUT_DIR = "outputs"
MODEL_DIR = "models"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
raw = pd.read_csv(DATA_PATH)
raw = raw.replace("?", np.nan)

y = (raw["readmitted"] == "<30").astype(int)
missing_pct = (raw.isnull().sum() / len(raw) * 100).sort_values(ascending=False)

drop_cols = ["encounter_id", "patient_nbr", "readmitted"] + missing_pct[missing_pct > 80].index.tolist()
X = raw.drop(columns=drop_cols)

print(raw.shape)
y.value_counts()

In [ ]:
missing_pct[missing_pct > 0]

In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat_cols,
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
y.value_counts().plot(kind="bar", ax=ax)
ax.set_title("target distribution")
ax.set_xticklabels(["no", "yes"], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "target_distribution.png"), dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
missing_pct[missing_pct > 0].plot(kind="barh", ax=ax)
ax.set_title("missing % by column")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "missing_values.png"), dpi=150)
plt.show()

logistic regression baseline, then random forest with grid search. using balanced class weights because the target is imbalanced.

In [ ]:
lr_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
lr_pipe.fit(X_train, y_train)
lr_test_pred = lr_pipe.predict(X_test)
lr_test_prob = lr_pipe.predict_proba(X_test)[:, 1]

lr_metrics = {
    "Accuracy": accuracy_score(y_test, lr_test_pred),
    "Precision": precision_score(y_test, lr_test_pred, zero_division=0),
    "Recall": recall_score(y_test, lr_test_pred, zero_division=0),
    "F1": f1_score(y_test, lr_test_pred, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, lr_test_prob),
}

In [ ]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
])

param_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [8, 12, 16],
    "clf__min_samples_split": [20, 50],
    "clf__min_samples_leaf": [10, 20],
}

grid = GridSearchCV(rf_pipe, param_grid, scoring="roc_auc", cv=5, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
grid.best_params_, grid.best_score_

In [ ]:
best_rf = grid.best_estimator_
best_params = {k.replace("clf__", ""): v for k, v in grid.best_params_.items()}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_rows = []

for fold_num, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    fold_model = Pipeline([
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1, **best_params)),
    ])
    fold_model.fit(X_tr, y_tr)

    tr_pred = fold_model.predict(X_tr)
    tr_prob = fold_model.predict_proba(X_tr)[:, 1]
    val_pred = fold_model.predict(X_val)
    val_prob = fold_model.predict_proba(X_val)[:, 1]

    fold_rows.append({
        "fold": fold_num,
        "train_accuracy": accuracy_score(y_tr, tr_pred),
        "train_precision": precision_score(y_tr, tr_pred, zero_division=0),
        "train_recall": recall_score(y_tr, tr_pred, zero_division=0),
        "train_f1": f1_score(y_tr, tr_pred, zero_division=0),
        "train_roc_auc": roc_auc_score(y_tr, tr_prob),
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_precision": precision_score(y_val, val_pred, zero_division=0),
        "val_recall": recall_score(y_val, val_pred, zero_division=0),
        "val_f1": f1_score(y_val, val_pred, zero_division=0),
        "val_roc_auc": roc_auc_score(y_val, val_prob),
    })

fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(os.path.join(OUTPUT_DIR, "cv_fold_results.csv"), index=False)
fold_df.round(4)

In [ ]:
summary = pd.DataFrame({
    "train_mean": [fold_df[f"train_{m}"].mean() for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]],
    "train_std": [fold_df[f"train_{m}"].std() for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]],
    "val_mean": [fold_df[f"val_{m}"].mean() for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]],
    "val_std": [fold_df[f"val_{m}"].std() for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]],
}, index=["accuracy", "precision", "recall", "f1", "roc_auc"])
summary.round(4)

In [ ]:
best_rf.fit(X_train, y_train)
rf_test_pred = best_rf.predict(X_test)
rf_test_prob = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_test_pred))

rf_metrics = {
    "Accuracy": accuracy_score(y_test, rf_test_pred),
    "Precision": precision_score(y_test, rf_test_pred, zero_division=0),
    "Recall": recall_score(y_test, rf_test_pred, zero_division=0),
    "F1": f1_score(y_test, rf_test_pred, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, rf_test_prob),
}
cm = confusion_matrix(y_test, rf_test_pred)
cm

In [ ]:
comparison = pd.DataFrame([lr_metrics, rf_metrics], index=["Logistic Regression", "Random Forest"])
comparison.round(4)

In [ ]:
comparison.round(4).to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"))
joblib.dump(best_rf, os.path.join(MODEL_DIR, "random_forest_best.pkl"))
joblib.dump(lr_pipe, os.path.join(MODEL_DIR, "logistic_regression.pkl"))

In [ ]:
metrics_list = ["Accuracy", "Precision", "Recall", "F1", "ROC AUC"]
x = np.arange(len(metrics_list))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, [lr_metrics[m] for m in metrics_list], width, label="log reg")
ax.bar(x + width/2, [rf_metrics[m] for m in metrics_list], width, label="random forest")
ax.set_xticks(x)
ax.set_xticklabels(metrics_list)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "rf_confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_test_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_test_prob)
ax.plot(fpr_lr, tpr_lr, label=f"log reg ({lr_metrics['ROC AUC']:.3f})")
ax.plot(fpr_rf, tpr_rf, label=f"rf ({rf_metrics['ROC AUC']:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_comparison.png"), dpi=150)
plt.show()

In [ ]:
feature_names = best_rf.named_steps["prep"].get_feature_names_out()
imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": best_rf.named_steps["clf"].feature_importances_,
}).sort_values("importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=imp_df, y="feature", x="importance", ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_importance.png"), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fold_df["fold"], fold_df["train_roc_auc"], "o-", label="train")
ax.plot(fold_df["fold"], fold_df["val_roc_auc"], "s-", label="val")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cv_fold_performance.png"), dpi=150)
plt.show()

In [ ]:
auc_gap = fold_df["train_roc_auc"].mean() - fold_df["val_roc_auc"].mean()
best_name = "Random Forest" if rf_metrics["ROC AUC"] >= lr_metrics["ROC AUC"] else "Logistic Regression"
top = imp_df.head(5)["feature"].str.replace("cat__", "").str.replace("num__", "").tolist()

print(f"best model: {best_name}")
print(f"test roc auc: {max(rf_metrics['ROC AUC'], lr_metrics['ROC AUC']):.4f}")
print(f"train-val auc gap: {auc_gap:.4f}")
print(f"top features: {top}")